# Fig. S7D: Gene expression changes associated with Hepatovirus infection in chicken liver samples, assessed by GSEA

Additional BLASTx top-hit tables used to refine negative controls.
Run `FigS7D_taxonkit.sh` at the indicated step; the resulting sample labels are used by `FigS7D_DESeq2.R`.


In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
# Define paths
basedir = "/path/to/project"
directory_out = "FigS7D"

# Output directory
path_out = f"{basedir}/output/revise_final/{directory_out}/blastx"
phage_path = f"{path_out}/phage_list.txt"
hepatometa_out_dir = f"{basedir}/output/revise_final/{directory_out}/metadata"

# Input files
blacklist_path = f"{basedir}/data/black_list_TN_ISG_low_new_format.txt"
# BLASTx results for RNA-seq samples
path_in = f"{basedir}/data/diamond/blast_out"
# Sample metadata
hepatometa = f"{basedir}/data/251215_rna_seq/revise_hepatovirus/hepatovirus_metadata_edited.tsv"
# BLASTx validation sample list
blastinfo = f"{basedir}/data/diamond/output/SraRunTable_blastx_from_logan_PRJNA774885_check.txt"

os.makedirs(path_out, exist_ok=True)
os.makedirs(hepatometa_out_dir, exist_ok=True)
print("saving files:", path_out)


saving files: /Users/coco/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/revise_final/FigS7D/blastx


In [3]:
def get_all_taxids(root_dir, takonkit_path):
    """
    Collect taxids from BLASTx top hits across RNA-seq samples.
    """
    # Initialize the taxid set
    taxids = set()

    # Read BLASTx hits from each sample
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if not (filename.endswith(".txt")):  # Skip non-result files
                continue

            # Path to the BLAST result file
            full_path = os.path.join(dirpath, filename)

            # Read BLAST hits
            col_name = [
                "qseqid", "sseqid", "pident", "length", "mismatch",
                "gapopen", "qstart", "qend", "sstart", "send", "evalue", 
                "bitscore", "qframe", "staxids", "stitle", "qlen", 
                "slen",
                "qcovhsp", "scovhsp"
            ]

            if os.path.getsize(full_path) == 0:
                print("file size 0 ")
                continue

            one_sample = pd.read_table(full_path, header=None, dtype={13:str})  # Read taxids as strings
            one_sample.columns = col_name

            # Check that every hit has a valid bit score
            if one_sample['bitscore'].isna().sum() > 0:
                print("NaN", filename)
            if (one_sample['bitscore'].dtype != np.float64) and (one_sample['bitscore'].dtype != int):
                print("dtype", filename, one_sample['bitscore'].dtype)
                print(one_sample['bitscore'])

            
            # Add the sample ID when absent from the query ID
            title = filename.strip().split("_")[0]
            if title in ["ERR11630559", "ERR11639626", "ERR11639659"]:
                one_sample["qseqid"] = one_sample["qseqid"].apply(lambda x: f"{title}_{x}")
        
            # Sort hits by bit score
            one_sample_sored = one_sample.sort_values("bitscore", ascending=False)

            # Remove hits without taxids
            one_sample_sored = one_sample_sored.dropna(subset=["staxids"])

            # Keep the top hit for each query
            one_sample_uniq = one_sample_sored.groupby("qseqid").first()
            
            # Collect taxids
            for line_taxid in one_sample_uniq["staxids"].to_list():
                if (line_taxid is None) or (line_taxid!=line_taxid):
                    continue
                raw_taxids = line_taxid.split(";")
                for i in range(len(raw_taxids)):
                    taxids.add(raw_taxids[i])
    
    # Write one taxid per line
    with open(takonkit_path, "w") as f:
        for taxid in sorted(list(taxids)):
            f.write(taxid + "\n")
    print(f"{len(taxids)} taxids in total")

    return taxids


def get_viral_hits(input_file, out_dir, lineage_info, blacklst):
    """
    Annotate top hits and exclude phages and blacklisted viral families.
    """
    # Count hits spanning multiple taxonomic groups
    num_ambiguous_hit = 0

    # Output file path
    output_folder = out_dir + "/taxonomy_output/"
    if not(os.path.exists(output_folder)):
        os.mkdir(output_folder)

    output_all_path = output_folder + "/" + input_file.split("/")[-1].split(".")[0] + ".blastx.taxonomy.txt"
    output_viral_path = output_folder + "/" + input_file.split("/")[-1].split(".")[0] + ".blastx.virus.txt"
    output_viral_black_path = output_folder + "/" + input_file.split("/")[-1].split(".")[0] + ".blastx.virus.black.txt"

    # Read BLAST hits
    col_name = [
        "qseqid", "sseqid", "pident", "length", "mismatch",
        "gapopen", "qstart", "qend", "sstart", "send", "evalue", 
        "bitscore", "qframe", "staxids", "stitle", "qlen", "slen",  
        "qcovhsp", "scovhsp"
    ]

    one_sample = pd.read_table(input_file, header=None, dtype={13:str})
    one_sample.columns = col_name

    # Add the sample ID when absent from the query ID
    title = input_file.strip().split("/")[-1].split("_")[0]
    if title in ["ERR11630559", "ERR11639626", "ERR11639659"]:
        one_sample["qseqid"] = one_sample["qseqid"].apply(lambda x: f"{title}_{x}")
        
    # Sort hits by bit score
    one_sample_sored = one_sample.sort_values("bitscore", ascending=False)

    # Remove hits without taxids
    one_sample_sored = one_sample_sored.dropna(subset=["staxids"])

    # Keep the top hit for each query
    one_sample_uniq = one_sample_sored.groupby("qseqid").first()

    # Count all contigs
    num_all_hits = len(set(one_sample_uniq.index.to_list()))

    # Add taxonomic annotations
    one_sample_tax = one_sample_uniq.copy()
    one_sample_tax["staxids_lst"] = one_sample_tax["staxids"].apply(lambda x: x.split(";") if((x is not None) and (x==x)) else x)
    one_sample_tax["name_lst"] = one_sample_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "fgs"] for taxid in x] if((x is not None) and (x==x)) else x)
    one_sample_tax["group_lst"] = one_sample_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "domain"] for taxid in x] if((x is not None) and (x==x)) else x)

    # Check for mixed viral and non-viral taxonomic groups
    for idx in one_sample_tax.index:
        groups = one_sample_tax.loc[idx, "group_lst"]
        if ((groups is not None) and (groups == groups)) and any("Viruses" in item for item in groups) and (len(set(groups)) > 1):
            num_ambiguous_hit += 1

    one_sample_tax["taxid_blastx"] = one_sample_tax["staxids_lst"].apply(lambda x: x[0] if((x is not None) and (x==x)) else x)
    one_sample_tax["family"] = one_sample_tax["staxids_lst"].apply(lambda x: lineage_info.loc[x[0], "family"] if((x is not None) and (x==x)) else x)
    one_sample_tax["genus"] = one_sample_tax["staxids_lst"].apply(lambda x: lineage_info.loc[x[0], "genus"] if((x is not None) and (x==x)) else x)
    one_sample_tax["species"] = one_sample_tax["staxids_lst"].apply(lambda x: lineage_info.loc[x[0], "species"] if((x is not None) and (x==x)) else x)

    # Mark hits as viral only when all taxonomic groups are viral
    one_sample_tax["is_virus"] = one_sample_tax["group_lst"].apply(lambda x: 1 if((x is not None) and (x==x)) and all("Viruses" in item for item in x) else 0)

    # Retain viral hits
    one_sample_virus = one_sample_tax.copy()
    one_sample_virus = one_sample_virus[one_sample_virus["is_virus"] == 1]

    # Count viral contigs
    num_viral_hits = len(set(one_sample_virus.index.to_list()))

    # Read blacklisted viral families
    blackvirus = []
    with open(blacklst, "r") as f_in:
        for line in f_in:
            blackvirus.append(line.strip())

    # Exclude blacklisted viral families
    one_sample_black_removed = one_sample_virus.copy()
    one_sample_black_removed = one_sample_black_removed[~one_sample_black_removed["family"].isin(blackvirus)]

    # Count viral contigs after blacklist filtering
    num_viral_hits_after_filter = len(set(one_sample_black_removed.index.to_list()))

    # Save results
    one_sample_tax.to_csv(output_all_path, sep="\t")
    one_sample_virus.to_csv(output_viral_path, sep="\t")
    one_sample_black_removed.to_csv(output_viral_black_path, sep="\t")

    return one_sample_black_removed, num_all_hits, num_viral_hits, num_viral_hits_after_filter, num_ambiguous_hit

# TaxonKit

In [4]:
# Collect taxids from all samples
taxids = get_all_taxids(path_in, f"{path_out}/taxids.txt")

7211 taxids in total


Run `FigS7D_taxonkit.sh` in the taxonkit environment.

# Identify phages

In [5]:
# Read TaxonKit output
taxinfo = pd.read_table(f"{path_out}/taxids_long.txt", header=None, dtype={0:str})
taxinfo.columns = ["taxid", "lineage"]

# Split the lineage into taxonomic ranks
taxinfo["domain"] = taxinfo["lineage"].apply(lambda x: x.split(";")[0] if x == x else x)
taxinfo["kingdom"] = taxinfo["lineage"].apply(lambda x: x.split(";")[1] if x == x else x)
taxinfo["phylum"] = taxinfo["lineage"].apply(lambda x: x.split(";")[2] if x == x else x)
taxinfo["class"] = taxinfo["lineage"].apply(lambda x: x.split(";")[3] if x == x else x)
taxinfo["order"] = taxinfo["lineage"].apply(lambda x: x.split(";")[4] if x == x else x)
taxinfo["family"] = taxinfo["lineage"].apply(lambda x: x.split(";")[5] if x == x else x)
taxinfo["genus"] = taxinfo["lineage"].apply(lambda x: x.split(";")[6] if x == x else x)
taxinfo["species"] = taxinfo["lineage"].apply(lambda x: x.split(";")[7] if x == x else x)

# Check missing values
print("#NaN:", taxinfo.isna().sum().sum())
# Fill missing values
taxinfo = taxinfo.fillna("-")

taxinfo

#NaN: 36


,taxid,lineage,domain,kingdom,phylum,class,order,family,genus,species
0,100035,Eukaryota;Fungi;Ascomycota;Dothideomycetes;Ple...,Eukaryota,Fungi,Ascomycota,Dothideomycetes,Pleosporales,unclassified Pleosporales family,Massariosphaeria,Massariosphaeria phaeospora
1,1000568,Bacteria;Bacillati;Bacillota;Negativicutes;Vei...,Bacteria,Bacillati,Bacillota,Negativicutes,Veillonellales,Veillonellaceae,Megasphaera,Megasphaera lornae
2,1000590,Bacteria;Bacillati;Bacillota;Bacilli;Bacillale...,Bacteria,Bacillati,Bacillota,Bacilli,Bacillales,Staphylococcaceae,Staphylococcus,Staphylococcus epidermidis
3,100182,Eukaryota;Metazoa;Chordata;Mammalia;Lagomorpha...,Eukaryota,Metazoa,Chordata,Mammalia,Lagomorpha,Leporidae,Lepus,Lepus granatensis
4,10020,Eukaryota;Metazoa;Chordata;Mammalia;Rodentia;H...,Eukaryota,Metazoa,Chordata,Mammalia,Rodentia,Heteromyidae,Dipodomys,Dipodomys ordii
...,...,...,...,...,...,...,...,...,...,...
7206,999415,Bacteria;Bacillati;Bacillota;Erysipelotrichia;...,Bacteria,Bacillati,Bacillota,Erysipelotrichia,Erysipelotrichales,Coprobacillaceae,Eggerthia,Eggerthia catenaformis
7207,9995,Eukaryota;Metazoa;Chordata;Mammalia;Rodentia;S...,Eukaryota,Metazoa,Chordata,Mammalia,Rodentia,Sciuridae,Marmota,Marmota monax
7208,999892,Bacteria;Bacillati;Actinomycetota;Actinomycete...,Bacteria,Bacillati,Actinomycetota,Actinomycetes,Propionibacteriales,Propionibacteriaceae,Cutibacterium,Cutibacterium modestum
7209,9999,Eukaryota;Metazoa;Chordata;Mammalia;Rodentia;S...,Eukaryota,Metazoa,Chordata,Mammalia,Rodentia,Sciuridae,Urocitellus,Urocitellus parryii


In [6]:
# Check table dimensions
print("Original size:", taxinfo.shape[0])

# Identify phage taxa
taxinfo["is_phage"] = 0
taxinfo["r1_Caudoviricetes"] = 0
taxinfo["r2_includes_phage"] = 0
taxinfo["r3_Inoviridae"] = 0
taxinfo["r4_Lavidaviridae"] = 0
taxinfo["r5_Microviridae"] = 0
phage_group_order = set()

for idx in taxinfo.index:
    # 1. Caudoviricetes
    if taxinfo.loc[idx, "class"] == "Caudoviricetes":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r1_Caudoviricetes"] = 1
    # 2. Unclassified viral species containing "phage"
    if ("phage" in taxinfo.loc[idx, "species"]) and (taxinfo.loc[idx, "domain"] == 'unclassified Viruses domain'):
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r2_includes_phage"] = 1
        phage_group_order.add(taxinfo.loc[idx, "order"])
    # 3. Inoviridae
    if taxinfo.loc[idx, "family"] == "Inoviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r3_Inoviridae"] = 1
    # 4. Lavidaviridae
    if taxinfo.loc[idx, "family"] == "Lavidaviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r4_Lavidaviridae"] = 1
    # 5. Microviridae
    if taxinfo.loc[idx, "family"] == "Microviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r5_Microviridae"] = 1

# Summarize phage orders
phage_group_order = sorted(list(phage_group_order))
print("phage order:", phage_group_order)

# Check table dimensions
print("Caudoviricetes:", taxinfo.loc[taxinfo["r1_Caudoviricetes"] == 1, :].shape[0])
print("phage:", taxinfo.loc[taxinfo["r2_includes_phage"] == 1, :].shape[0])
print(" * not Caudoviricetes:", taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :].shape[0])
print("Inoviridae:", taxinfo.loc[taxinfo["r3_Inoviridae"] == 1, :].shape[0])
print("Lavidaviridae:", taxinfo.loc[taxinfo["r4_Lavidaviridae"] == 1, :].shape[0])
print("Microviridae:", taxinfo.loc[taxinfo["r5_Microviridae"] == 1, :].shape[0])

# Save the annotation table
taxinfo.to_csv(os.path.join(path_out, "taxids_phage.tsv"), sep="\t")

# Save the phage taxid list
phages = taxinfo.loc[taxinfo["is_phage"] == 1, "taxid"].to_list()
with open(phage_path, "w") as f:
    for pid in phages:
        f.write(pid + "\n")

# Preview results
taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :]

Original size: 7211
phage order: ['Crassvirales', 'unclassified Caudoviricetes order', 'unclassified Viruses order']
Caudoviricetes: 168
phage: 101
 * not Caudoviricetes: 3
Inoviridae: 0
Lavidaviridae: 0
Microviridae: 0


,taxid,lineage,domain,kingdom,phylum,class,order,family,genus,species,is_phage,r1_Caudoviricetes,r2_includes_phage,r3_Inoviridae,r4_Lavidaviridae,r5_Microviridae
4387,2938905,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,Staphylococcus phage KMSP1,1,0,1,0,0,0
5207,38018,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,Bacteriophage sp.,1,0,1,0,0,0
6388,707152,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,uncultured marine phage,1,0,1,0,0,0


# Add taxonomy

In [7]:
# Add taxonomic annotations to the taxid list
taxid_in = f"{path_out}/taxids_lineage.txt"

# Read the lineage table
lineage_info = pd.read_table(taxid_in, header=None)
lineage_info.columns = ["taxid", "domain", "fgs"]

# Check missing values
print("#NaN beginning:", lineage_info.isna().sum().sum())
nan_rows =lineage_info[lineage_info.isna().any(axis=1)]
print(nan_rows)
print()

# Add family, genus and species columns
lineage_info["taxid"] = lineage_info["taxid"].astype(str)
lineage_info = lineage_info.set_index("taxid")
lineage_info["family"] = lineage_info["fgs"].apply(lambda x: x.split(";")[0] if x==x else np.nan)  
lineage_info["genus"] = lineage_info["fgs"].apply(lambda x: x.split(";")[1] if x==x else np.nan)  
lineage_info["species"] = lineage_info["fgs"].apply(lambda x: x.split(";")[2] if x==x else np.nan)  

# Label phage taxa as phage_group
phage_ids = []
with open(phage_path, "r") as f:
    for line in f:
        line = line.strip()
        phage_ids.append(line)

for idx in lineage_info.index:
    if idx in phage_ids:
        lineage_info.loc[idx, "domain"] = "phage_group"

# Check missing values
print("#NaN end:", lineage_info.isna().sum().sum())
nan_rows =lineage_info[lineage_info.isna().any(axis=1)]
print(nan_rows)
print()

# Fill missing values
lineage_info = lineage_info.fillna("-")

print(f"{lineage_info.shape[0]} taxids in total\n")

# Preview hits from each domain group
print("All domain:", set(lineage_info["domain"].to_list()), "\n")
for domain in set(lineage_info["domain"].to_list()):
    print(f"##### {domain} ####")
    print(f"size: {lineage_info[lineage_info["domain"] == domain].shape[0]}")
    print(lineage_info[["domain", "species"]][lineage_info["domain"] == domain].head(3))
    print()

# Preview results
lineage_info

#NaN beginning: 8
        taxid domain  fgs
5003  3427865    NaN  NaN
5030  3466922    NaN  NaN
5150  3702686    NaN  NaN
5193  3774379    NaN  NaN

#NaN end: 20
        domain  fgs family genus species
taxid                                   
3427865    NaN  NaN    NaN   NaN     NaN
3466922    NaN  NaN    NaN   NaN     NaN
3702686    NaN  NaN    NaN   NaN     NaN
3774379    NaN  NaN    NaN   NaN     NaN

7211 taxids in total

All domain: {'phage_group', 'unclassified other entries domain', 'Archaea', 'Bacteria', 'unclassified Viruses domain', '-', 'Eukaryota'} 

##### phage_group ####
size: 171
              domain              species
taxid                                    
1129194  phage_group     Dibbivirus DIBBI
1211417  phage_group  Carjivirus communis
1229751  phage_group  Vedamuthuvirus BM13

##### unclassified other entries domain ####
size: 256
                                    domain  \
taxid                                        
111789   unclassified other entries dom

,domain,fgs,family,genus,species
taxid,,,,,
100035,Eukaryota,unclassified Pleosporales family;Massariosphae...,unclassified Pleosporales family,Massariosphaeria,Massariosphaeria phaeospora
1000568,Bacteria,Veillonellaceae;Megasphaera;Megasphaera lornae,Veillonellaceae,Megasphaera,Megasphaera lornae
1000590,Bacteria,Staphylococcaceae;Staphylococcus;Staphylococcu...,Staphylococcaceae,Staphylococcus,Staphylococcus epidermidis
100182,Eukaryota,Leporidae;Lepus;Lepus granatensis,Leporidae,Lepus,Lepus granatensis
10020,Eukaryota,Heteromyidae;Dipodomys;Dipodomys ordii,Heteromyidae,Dipodomys,Dipodomys ordii
...,...,...,...,...,...
999415,Bacteria,Coprobacillaceae;Eggerthia;Eggerthia catenaformis,Coprobacillaceae,Eggerthia,Eggerthia catenaformis
9995,Eukaryota,Sciuridae;Marmota;Marmota monax,Sciuridae,Marmota,Marmota monax
999892,Bacteria,Propionibacteriaceae;Cutibacterium;Cutibacteri...,Propionibacteriaceae,Cutibacterium,Cutibacterium modestum


# Extract viral top hits

In [8]:
def count_viral_contigs(path_in: str, infection_th=1):
    """
    Collect filtered viral contigs across RNA-seq samples.
    """

    # Initialize the ambiguous-hit count and first-file flag
    num_amb_hit = 0
    is_fist_file = 1

    # Read all BLAST result files
    for dirpath, dirnames, filenames in os.walk(path_in):
        for filename in filenames:
            if not (filename.endswith(".txt")):  # Skip non-result files
                continue

            # Path to the BLAST result file
            full_path = os.path.join(dirpath, filename)

            if os.path.getsize(full_path) == 0:
                print("filesize 0")
                continue

            # Annotate and save viral contigs
            df_one_sample, num_all_contigs, num_viral_contigs, num_viral_black, num_amb_hit_each = get_viral_hits(full_path, path_out, lineage_info, blacklist_path)
            num_amb_hit += num_amb_hit_each

            # Add the sample ID
            sample_name = filename.split("_")[0].strip()
            df_one_sample["sample"] = sample_name

            # Combine sample results
            if is_fist_file == 1:
                df_all = df_one_sample.copy()
                is_fist_file = 0
            else:
                df_all = pd.concat([df_all, df_one_sample])


    print("Virus ambiguous hit:", num_amb_hit)
    print("=======")

    return df_all

In [9]:
# Extract viral top hits
infection_threshod = 1
df_all = count_viral_contigs(path_in, infection_threshod)

Virus ambiguous hit: 852


In [10]:
# Create the viral contig annotation table
df = df_all.copy().reset_index()
print("shape:", df.shape)
# Add viral contig counts per sample
virus_hit_count = pd.DataFrame(df.value_counts("sample"))
df = df.merge(virus_hit_count, on="sample", how="left")
df = df.sort_values("sample")
df = df.sort_values("count", ascending=False)
# Check missing values
print("#NaN:", df.isna().sum().sum())
df.to_csv(f"{path_out}/virus_all.txt", sep="\t")
# Preview results
df.head(5)

shape: (1991, 28)
#NaN: 0


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,...,staxids_lst,name_lst,group_lst,taxid_blastx,family,genus,species,is_virus,sample,count
92,ERR11639811_936189,QLC35796.1,100.0,79,0,0,239,3,1216,1294,...,"[31631, 2749450]",[Coronaviridae;Betacoronavirus;Betacoronavirus...,"[unclassified Viruses domain, unclassified Vir...",31631,Coronaviridae,Betacoronavirus,Betacoronavirus 1,1,ERR11639811,55
59,ERR11639811_375337,AUF73151.1,94.9,99,5,0,299,3,571,669,...,[694448],[Coronaviridae;unclassified Coronaviridae genu...,[unclassified Viruses domain],694448,Coronaviridae,unclassified Coronaviridae genus,unidentified human coronavirus,1,ERR11639811,55
81,ERR11639811_734333,QLC35802.1,100.0,82,0,0,3,248,2398,2479,...,[2749448],[Coronaviridae;unclassified Coronaviridae genu...,[unclassified Viruses domain],2749448,Coronaviridae,unclassified Coronaviridae genus,Coronavirus cya-BetaCoV/2019,1,ERR11639811,55
80,ERR11639811_731099,QLC35790.1,100.0,78,0,0,3,236,7018,7095,...,[2749449],[Coronaviridae;unclassified Coronaviridae genu...,[unclassified Viruses domain],2749449,Coronaviridae,unclassified Coronaviridae genus,Coronavirus cyb-BetaCoV/2019,1,ERR11639811,55
79,ERR11639811_706779,YP_009555243.1,100.0,59,0,0,216,40,26,84,...,[31631],[Coronaviridae;Betacoronavirus;Betacoronavirus 1],[unclassified Viruses domain],31631,Coronaviridae,Betacoronavirus,Betacoronavirus 1,1,ERR11639811,55


# Identify virus-positive samples

In [11]:
# List unique samples for each viral taxonomic group
picornaviridae_samples = sorted(
    df.loc[df["family"].eq("Picornaviridae"), "sample"].dropna().unique()
)
print(f"Picornaviridae samples ({len(picornaviridae_samples)}):")
print(sorted(picornaviridae_samples))

hepatovirus_samples = sorted(
    df.loc[df["genus"].str.casefold().eq("hepatovirus"), "sample"].dropna().unique()
)
print(f"Hepatovirus samples ({len(hepatovirus_samples)}):")
print(sorted(hepatovirus_samples))

Picornaviridae samples (22):
['ERR11630658', 'ERR11630659', 'ERR11630670', 'ERR11630679', 'ERR11630685', 'ERR11634336', 'ERR11634383', 'ERR11634418', 'ERR11634459', 'SRR16585085', 'SRR16585086', 'SRR16585088', 'SRR16585089', 'SRR16585097', 'SRR16585112', 'SRR16585113', 'SRR16585114', 'SRR16585115', 'SRR16585116', 'SRR16585117', 'SRR16585118', 'SRR16585120']
Hepatovirus samples (13):
['ERR11630658', 'ERR11630659', 'ERR11630670', 'ERR11630679', 'ERR11630685', 'ERR11634383', 'ERR11634418', 'SRR16585086', 'SRR16585114', 'SRR16585116', 'SRR16585117', 'SRR16585118', 'SRR16585120']


# Update sample metadata

In [12]:
# Read sample metadata, preserving empty cells and existing values
hepatometa_df = pd.read_table(hepatometa, dtype=str, keep_default_na=False)

# Set Negative to 0 for samples with Picornaviridae hits
hepatometa_family = hepatometa_df.copy()
family_mask = hepatometa_family["Sample"].isin(picornaviridae_samples)
hepatometa_family.loc[family_mask, "Negative"] = "0"
family_output_path = os.path.join(hepatometa_out_dir, "hepatpviurs_metadata_family.tsv")
hepatometa_family.to_csv(family_output_path, sep="\t", index=False)
print(f"Picornaviridae: Negative = 0 for {family_mask.sum()} rows; saved to {family_output_path}")

# Set Negative to 0 for samples with Hepatovirus hits in a separate table
hepatometa_genus = hepatometa_df.copy()
genus_mask = hepatometa_genus["Sample"].isin(hepatovirus_samples)
hepatometa_genus.loc[genus_mask, "Negative"] = "0"
genus_output_path = os.path.join(hepatometa_out_dir, "hepatovirus_metadata_genus.tsv")
hepatometa_genus.to_csv(genus_output_path, sep="\t", index=False)
print(f"Hepatovirus: Negative = 0 for {genus_mask.sum()} rows; saved to {genus_output_path}")

Picornaviridae: Negative = 0 for 7 rows; saved to /Users/coco/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/revise_final/FigS7D/metadata/hepatpviurs_metadata_family.tsv
Hepatovirus: Negative = 0 for 5 rows; saved to /Users/coco/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/revise_final/FigS7D/metadata/hepatovirus_metadata_genus.tsv


# Check samples without contigs

In [13]:
# List PRJNA774885 samples without contigs
blastinfo_df = pd.read_table(blastinfo, dtype=str, keep_default_na=False)
no_contig_mask = (
    blastinfo_df["BioProject"].eq("PRJNA774885")
    & blastinfo_df["contig"].str.strip().str.casefold().eq("false")
)
no_contig_samples = sorted(blastinfo_df.loc[no_contig_mask, "Run"].unique())

# Match Run IDs to Sample IDs in the metadata
hepatometa_for_contig_check = pd.read_table(hepatometa, dtype=str, keep_default_na=False)
no_contig_metadata = hepatometa_for_contig_check.loc[
    hepatometa_for_contig_check["BioProject"].eq("PRJNA774885")
    & hepatometa_for_contig_check["Sample"].isin(no_contig_samples)
]
positive_no_contig_samples = sorted(
    no_contig_metadata.loc[
        pd.to_numeric(no_contig_metadata["Positive"], errors="coerce").eq(1), "Sample"
    ].unique()
)
negative_no_contig_samples = sorted(
    no_contig_metadata.loc[
        pd.to_numeric(no_contig_metadata["Negative"], errors="coerce").eq(1), "Sample"
    ].unique()
)

print(f"PRJNA774885 contig=False samples: {len(no_contig_samples)}")
print(f"contig=False and Positive=1 ({len(positive_no_contig_samples)}): {positive_no_contig_samples}")
print(f"contig=False and Negative=1 ({len(negative_no_contig_samples)}): {negative_no_contig_samples}")

PRJNA774885 contig=False samples: 19
contig=False and Positive=1 (1): ['SRR16585087']
contig=False and Negative=1 (0): []
